# agent_framework_tools

## Example 1: run a command in a stateless local shell

In [ ]:
# Example 1: LocalShellTool runs `echo` and returns its exit code and output
import tempfile
from agent_framework_tools.shell import LocalShellTool

shell = LocalShellTool(mode="stateless", workdir=tempfile.mkdtemp(), acknowledge_unsafe=True, approval_mode="never_require")
result = await shell.run("echo hello from the shell")
assert result.exit_code == 0 and "hello from the shell" in result.stdout, result
print("Example 1:", result.exit_code, result.stdout.strip())
await shell.close()

## Example 2: a persistent shell keeps state between commands

## Example 3: a deny policy refuses a command before it runs

In [ ]:
# Example 3: ShellPolicy's denylist blocks `rm`; the tool raises instead of running it
import tempfile
from agent_framework_tools.shell import LocalShellTool, ShellCommandError, ShellPolicy

shell = LocalShellTool(mode="stateless", workdir=tempfile.mkdtemp(), acknowledge_unsafe=True, approval_mode="never_require",
                       policy=ShellPolicy(denylist=[r"\brm\b"]))
try:
    await shell.run("rm -rf ./anything")
except ShellCommandError as error:
    print("Example 3: refused by policy:", str(error)[:80])
else:
    raise AssertionError("a denylisted command ran")
await shell.close()

## Example 4: a command that outlives its timeout is stopped

In [ ]:
# Example 4: a 1-second timeout stops `sleep 30`
import tempfile, time
from agent_framework_tools.shell import LocalShellTool, ShellTimeoutError

shell = LocalShellTool(mode="stateless", workdir=tempfile.mkdtemp(), acknowledge_unsafe=True, approval_mode="never_require", timeout=1.0)
start = time.monotonic()
try:
    result = await shell.run("sleep 30")
except ShellTimeoutError as error:
    print("Example 4: timed out as expected after", round(time.monotonic() - start, 1), "s")
else:
    assert result.timed_out if hasattr(result, "timed_out") else False, result
    print("Example 4: result reports a timeout after", round(time.monotonic() - start, 1), "s")
assert time.monotonic() - start < 15
await shell.close()

## Example 5: the shell as an agent tool

In [ ]:
# Example 5: as_function() wraps the shell as a FunctionTool an Agent can call
import tempfile
from agent_framework import FunctionTool
from agent_framework_tools.shell import LocalShellTool

shell = LocalShellTool(mode="stateless", workdir=tempfile.mkdtemp(), acknowledge_unsafe=True, approval_mode="never_require")
function = shell.as_function()
assert isinstance(function, FunctionTool), type(function)
print("Example 5:", function.name, "-", (function.description or "")[:60])
await shell.close()